Aggregate per-year storm exceedance metrics (from `storm_percentile_metrics.py`) over the full 35-year period for each simulation. Storms spanning two calendar years are summed.

In [1]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

In [2]:
simulations = ['UBD', 'UBE', 'UBF', 'UBG', 'UBH', 'UBI']
future_hist_sim = {'UBG': 'UBD', 'UBH': 'UBE', 'UBI': 'UBF'}
quantiles = [99.0, 99.9]
wetdays = False
future = False
mask = 'land'
region = 'NNA2'  # e.g. 'NNA' to restrict to the Northeastern North America box, or None for the global mask
tc_filter = 'phase'  # None (all cyclones) or 'phase' (no tropical/transition time steps)

In [3]:
base = '/home/vdemeyer/projects/rrg-gachon/vdemeyer'

quantile_tag = {99.0: '99', 99.9: '999'}
mtag = 'urbanonly' if mask == 'urban' else 'landonly'
region_tag = f'_{region}' if region is not None else ''
tc_tag = {None: '', 'phase': '_noTC'}[tc_filter]

for sim in simulations:

    # Determine period
    if sim in future_hist_sim:
        years = range(2063, 2098)
    else:
        years = range(1980, 2015)

    # Build suffix for filenames
    add_file = ''
    if wetdays:
        add_file += '_wetdays'
    if future and sim in future_hist_sim:
        add_file += '_future_percentile'    

    per_quantile_dfs = []

    for q in quantiles:
        tag = quantile_tag[q]

        # Load all per-year pickles for this quantile
        dfs = []
        for year in tqdm(years, desc=f'{sim} q={q}'):
            pkl_file = f'{base}/{sim}/STORM_RELATED/STORM_METRICS/storm_exceed_{tag}_metrics_{mtag}{region_tag}_{sim}_{year}{add_file}{tc_tag}.pkl'
            if not os.path.exists(pkl_file):
                print(f"Warning: missing {pkl_file}")
                continue
            dfs.append(pd.read_pickle(pkl_file))

        if not dfs:
            print(f"No pickle files found for {sim} q={q}, skipping")
            continue

        # Concatenate and sum cum/count for storms appearing in multiple years
        df_all = pd.concat(dfs, ignore_index=True)
        df_agg = df_all.groupby('storm_id', as_index=False).agg({
            'cum_excess_pr': 'sum',
            'count_exceed_pr': 'sum',
            'cum_excess_wind': 'sum',
            'count_exceed_wind': 'sum',
        })

        # Compute intensity = cum / count
        df_agg['mean_excess_pr'] = np.where(
            df_agg['count_exceed_pr'] > 0,
            df_agg['cum_excess_pr'] / df_agg['count_exceed_pr'],
            np.nan
        )
        df_agg['mean_excess_wind'] = np.where(
            df_agg['count_exceed_wind'] > 0,
            df_agg['cum_excess_wind'] / df_agg['count_exceed_wind'],
            np.nan
        )

        df_agg['quantile'] = q
        per_quantile_dfs.append(df_agg)

    if not per_quantile_dfs:
        continue

    df_agg = pd.concat(per_quantile_dfs, ignore_index=True)

    # Load tracking data and extract initiation date per storm
    storm_data_file = f'{base}/{sim}/STORM_RELATED/TRACK/{sim}_psl_smooth_400km_24h_1000hPa.txt'
    df_track = pd.read_csv(
        storm_data_file,
        sep=r' ', header=0, engine='python',
        names=['storm', 'point', 'i', 'j', 'date', 'lat', 'lon', 'pressure']
    )
    df_track['date'] = pd.to_datetime(df_track['date'])
    initiation = df_track.groupby('storm')['date'].min().reset_index()
    initiation.columns = ['storm_id', 'initiation_date']

    # Merge initiation date
    df_agg = df_agg.merge(initiation, on='storm_id', how='left')

    # Filter: keep only storms whose initiation falls within the period
    start_year, end_year = min(years), max(years)
    df_agg = df_agg[
        (df_agg['initiation_date'].dt.year >= start_year) &
        (df_agg['initiation_date'].dt.year <= end_year)
    ]

    # Reorder columns
    df_agg = df_agg[['storm_id', 'quantile', 'initiation_date',
                      'cum_excess_pr', 'count_exceed_pr', 'mean_excess_pr',
                      'cum_excess_wind', 'count_exceed_wind', 'mean_excess_wind']]
    df_agg = df_agg.sort_values(['quantile', 'storm_id']).reset_index(drop=True)

    # Save
    output_dir = f'{base}/{sim}/STORM_RELATED/STORM_METRICS'
    os.makedirs(output_dir, exist_ok=True)
    output_file = f'{output_dir}/storm_exceed_metrics_{mtag}{region_tag}_{sim}_{start_year}-{end_year}{add_file}{tc_tag}.pkl'
    df_agg.to_pickle(output_file)
    print(f"\n{sim}: saved {len(df_agg)} rows to {output_file}")
    for q in quantiles:
        sub = df_agg[df_agg['quantile'] == q]
        print(f"  q={q}: {len(sub)} storms | pr exceedance: {(sub['count_exceed_pr'] > 0).sum()} | wind exceedance: {(sub['count_exceed_wind'] > 0).sum()}")

UBD q=99.9: 100%|██████████| 35/35 [00:02<00:00, 17.15it/s]



UBD: saved 18544 rows to /home/vdemeyer/projects/rrg-gachon/vdemeyer/UBD/STORM_RELATED/STORM_METRICS/storm_exceed_metrics_landonly_NNA2_UBD_1980-2014_noTC.pkl
  q=99.0: 9272 storms | pr exceedance: 3896 | wind exceedance: 3617
  q=99.9: 9272 storms | pr exceedance: 2790 | wind exceedance: 2109


UBE q=99.9: 100%|██████████| 35/35 [00:01<00:00, 19.39it/s]



UBE: saved 18020 rows to /home/vdemeyer/projects/rrg-gachon/vdemeyer/UBE/STORM_RELATED/STORM_METRICS/storm_exceed_metrics_landonly_NNA2_UBE_1980-2014_noTC.pkl
  q=99.0: 9010 storms | pr exceedance: 3959 | wind exceedance: 3621
  q=99.9: 9010 storms | pr exceedance: 2707 | wind exceedance: 2113


UBF q=99.9: 100%|██████████| 35/35 [00:01<00:00, 21.03it/s]



UBF: saved 19896 rows to /home/vdemeyer/projects/rrg-gachon/vdemeyer/UBF/STORM_RELATED/STORM_METRICS/storm_exceed_metrics_landonly_NNA2_UBF_1980-2014_noTC.pkl
  q=99.0: 9948 storms | pr exceedance: 4215 | wind exceedance: 3731
  q=99.9: 9948 storms | pr exceedance: 3008 | wind exceedance: 2119


... (2 outputs left out)

UBH q=99.9: 100%|██████████| 35/35 [00:01<00:00, 23.52it/s]



UBH: saved 17734 rows to /home/vdemeyer/projects/rrg-gachon/vdemeyer/UBH/STORM_RELATED/STORM_METRICS/storm_exceed_metrics_landonly_NNA2_UBH_2063-2097_noTC.pkl
  q=99.0: 8867 storms | pr exceedance: 3815 | wind exceedance: 3554
  q=99.9: 8867 storms | pr exceedance: 2734 | wind exceedance: 2120


UBI q=99.9: 100%|██████████| 35/35 [00:01<00:00, 23.02it/s]



UBI: saved 19142 rows to /home/vdemeyer/projects/rrg-gachon/vdemeyer/UBI/STORM_RELATED/STORM_METRICS/storm_exceed_metrics_landonly_NNA2_UBI_2063-2097_noTC.pkl
  q=99.0: 9571 storms | pr exceedance: 4195 | wind exceedance: 3729
  q=99.9: 9571 storms | pr exceedance: 3084 | wind exceedance: 2067
